# Entrenamiento YOLOv8m-seg — Detector de Rayaduras

## Antes de ejecutar:
1. Panel derecho → **Add data** → sube `scratch_data.zip` como dataset
2. Panel derecho → **Settings** → **Accelerator**: GPU T4 x2
3. Panel derecho → **Settings** → **Internet**: On (requiere verificar teléfono en kaggle.com/settings)
4. Ejecuta las celdas en orden

In [ ]:
# CELDA 1 — Verificar GPU
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('AVISO: Sin GPU. Activa GPU T4 en Settings -> Accelerator')

In [ ]:
# CELDA 2 — Instalar ultralytics (necesita Internet ON)
# Si ya esta instalado no hace nada
try:
    from ultralytics import YOLO
    print('ultralytics ya instalado OK')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)
    from ultralytics import YOLO
    print('ultralytics instalado OK')

In [ ]:
# CELDA 3 — Localizar datos (zip o ya extraido)
import zipfile, os, shutil
from pathlib import Path

DATA_ROOT  = Path('/kaggle/working/scratch_detector/data')
TRAIN_IMGS = DATA_ROOT / 'images' / 'train'

# Caso A: datos ya extraidos en /kaggle/input (dataset subido como Kaggle dataset)
extracted = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dataset.yaml' in files:
        extracted = Path(root)
        break

if extracted and (extracted / 'images' / 'train').exists():
    print(f'Datos encontrados en: {extracted}')
    DATA_ROOT = extracted
else:
    # Caso B: buscar scratch_data.zip y extraer
    zip_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == 'scratch_data.zip':
                zip_path = Path(root) / f
                break
    if zip_path is None:
        raise FileNotFoundError('No se encontraron datos. Sube scratch_data.zip en Add data.')
    print(f'ZIP encontrado: {zip_path}  ({zip_path.stat().st_size/1e6:.0f} MB)')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/kaggle/working/scratch_detector/')

for split in ['train', 'val']:
    imgs = list((DATA_ROOT / 'images' / split).glob('*.jpg'))
    lbls = list((DATA_ROOT / 'labels' / split).glob('*.txt'))
    print(f'  {split}: {len(imgs)} imagenes, {len(lbls)} labels')

In [ ]:
# CELDA 4 — Fijar rutas en dataset.yaml
import yaml, shutil
from pathlib import Path

# DATA_ROOT viene de la celda anterior
YAML_SRC  = DATA_ROOT / 'dataset.yaml'
YAML_WORK = Path('/kaggle/working/dataset.yaml')

if YAML_SRC != YAML_WORK:
    shutil.copy2(YAML_SRC, YAML_WORK)

with open(YAML_WORK) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(DATA_ROOT)
with open(YAML_WORK, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('dataset.yaml OK:')
print(f"  path:  {cfg['path']}")
print(f"  nc:    {cfg['nc']}")
print(f"  names: {cfg['names']}")

In [ ]:
# CELDA 5 — ENTRENAMIENTO con YOLOv8m-seg
from ultralytics import YOLO
from pathlib import Path

YAML_PATH = Path('/kaggle/working/dataset.yaml')

model = YOLO('yolov8m-seg.pt')  # medium: 27M params vs 11M del small

results = model.train(
    data          = str(YAML_PATH),
    epochs        = 150,
    imgsz         = 640,
    batch         = 8,           # reducido por mayor VRAM del modelo m
    patience      = 30,
    lr0           = 0.01,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.1, scale=0.5,
    flipud=0.3,   fliplr=0.5,
    mosaic=1.0,   mixup=0.1,
    box=7.5, cls=1.5, dfl=1.5,
    project       = '/kaggle/working/runs',
    name          = 'scratch_yolo',
    save          = True,
    save_period   = 10,
    plots         = True,
    verbose       = True,
)
print('Entrenamiento completado')

In [ ]:
# CELDA 6 — Metricas finales
from ultralytics import YOLO
from pathlib import Path
import yaml

YAML_PATH = Path('/kaggle/working/dataset.yaml')
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

best = Path('/kaggle/working/runs/scratch_yolo/weights/best.pt')
model_best = YOLO(str(best))
metrics = model_best.val(data=str(YAML_PATH), split='val')

names = cfg.get('names', {})
print(f'Box  mAP50:    {metrics.box.map50:.4f}')
print(f'Box  mAP50-95: {metrics.box.map:.4f}')
print(f'Mask mAP50:    {metrics.seg.map50:.4f}')
print(f'Mask mAP50-95: {metrics.seg.map:.4f}')
for i, ap in enumerate(metrics.seg.ap50):
    print(f'  Mask AP50[{i}] {names.get(i, i)}: {ap:.4f}')

In [ ]:
# CELDA 7 — Ver ruta del modelo entrenado
# En Kaggle NO hay boton de descarga directa.
# Los archivos en /kaggle/working/ aparecen en el panel derecho -> Output
# Haz clic en best.pt para descargarlo.
from pathlib import Path

best = Path('/kaggle/working/runs/scratch_yolo/weights/best.pt')
last = Path('/kaggle/working/runs/scratch_yolo/weights/last.pt')

print('Archivos disponibles para descargar en Output:')
for p in [best, last]:
    if p.exists():
        print(f'  {p}  ({p.stat().st_size/1e6:.0f} MB)  <- descarga desde panel Output')
    else:
        print(f'  {p}  NO ENCONTRADO')

print('\nEn el panel derecho -> Output -> navega a runs/scratch_yolo/weights/')